# 20 — G2: controlled pose-model suite

This notebook keeps the accepted **V6-C gate and its frozen 0.675 internal threshold** unchanged. It changes only the supported-exercise classifier and compares three compact landmark models on the exact V6-A.1 split:

1. bidirectional GRU,
2. pose Transformer encoder,
3. adaptive graph convolution (CTR-GCN-inspired lightweight model).

It also reports an equal-probability ensemble selected without test-set weighting. The internal test is used for reporting, not hyperparameter tuning. A winner still needs Notebook 17-style creator-separated validation before promotion.


In [ ]:
%pip install -q scikit-learn seaborn pandas==2.2.3


In [ ]:
from pathlib import Path
import json
import random
import shutil
import urllib.request
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from google.colab import files
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from torch import nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

SEED = 42
WINDOW = 64
STRIDE = 32
BATCH_SIZE = 48
MAX_EPOCHS = 30
PATIENCE = 6
GROUP_TEMPERATURE = 0.5  # frozen from V6-A.1
AUGMENTATION_CONFIG = {
    "horizontal_flip_probability": 0.5,
    "rotation_probability": 0.7,
    "max_rotation_degrees": 7.0,
    "speed_probability": 0.5,
    "speed_range": [0.85, 1.15],
    "frame_drop_probability": 0.35,
    "frame_drop_fraction_range": [0.05, 0.12],
    "coordinate_noise_probability": 0.8,
    "coordinate_noise_std": 0.008,
}
RETIRED_CLASSES = {"plank"}
UNREVIEWED_GROUPS = {
    "youtube:oLsonHQJOUc",
    "youtube:c4TomZjzobY",
    "youtube:W2Bm9dfQjbk",
}


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("Experiment: G2 controlled pose-model suite")
assert DEVICE.type == "cuda", "Select a T4 runtime before training."


## Load the same frozen landmark evidence


In [ ]:
REPOSITORY_RAW = (
    "https://raw.githubusercontent.com/bariscelikk1/cALorie/"
    "feat/tcn-rebuild/ml/artifacts"
)
LANDMARK_ARCHIVES = [
    ("v1", "calorie_workoutfitness_landmarks_v1.zip"),
    ("v1", "calorie_mmfit_landmarks_v1.zip"),
    ("v1", "calorie_mmfit_unknown_v1.zip"),
    ("v2", "calorie_youtube_mixed_landmarks_v3.zip"),
    ("v3", "calorie_youtube_additional_landmarks_v1.zip"),
    ("v3", "calorie_youtube_training_landmarks_v4.zip"),
    ("v5", "calorie_workoutfitness_gate_hardneg_v1.zip"),
]


def download(url, destination):
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request, timeout=300) as response, destination.open("wb") as output:
        shutil.copyfileobj(response, output)
    print(f"{destination.name}: {destination.stat().st_size / 1e6:.1f} MB")


zip_paths = []
for version, name in LANDMARK_ARCHIVES:
    destination = Path("/content") / name
    download(f"{REPOSITORY_RAW}/{version}/landmarks/{name}", destination)
    zip_paths.append(destination)

HARD_POSITIVE_PATH = Path("/content/hard_positive_windows.csv")
download(
    f"{REPOSITORY_RAW}/v5/hard_positive_mining/hard_positive_windows.csv",
    HARD_POSITIVE_PATH,
)
hard_positive_frame = pd.read_csv(HARD_POSITIVE_PATH)
assert hard_positive_frame.window_key.is_unique, "Hard-positive window keys must be unique."

V6A1_SPLIT_PATH = Path("/content/v6a1_split_assignments.csv")
download(
    f"{REPOSITORY_RAW}/v6a1/model/split_assignments.csv",
    V6A1_SPLIT_PATH,
)

V6C_METRICS_PATH = Path("/content/v6c_metrics.json")
download(
    f"{REPOSITORY_RAW}/v6c/model/metrics.json",
    V6C_METRICS_PATH,
)
V6C_METRICS = json.loads(V6C_METRICS_PATH.read_text())

SOURCE_ROOT = Path("/content/g2_model_suite_landmark_sources")
if SOURCE_ROOT.exists():
    shutil.rmtree(SOURCE_ROOT)
SOURCE_ROOT.mkdir()
for index, archive_path in enumerate(zip_paths):
    destination = SOURCE_ROOT / f"source_{index}"
    destination.mkdir()
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(destination)

frames = []
for manifest_path in SOURCE_ROOT.rglob("manifest.csv"):
    frame = pd.read_csv(manifest_path).fillna("")
    frame["npz_path"] = frame["output_path"].map(
        lambda value: str(manifest_path.parent / value)
    )
    frames.append(frame)
assert frames, "No manifest.csv was found."
manifest = pd.concat(frames, ignore_index=True)
manifest = manifest[
    (manifest.status == "ok")
    & manifest.npz_path.map(lambda value: Path(value).exists())
].copy()
if "group_id" not in manifest or manifest.group_id.eq("").any():
    manifest["group_id"] = manifest.apply(
        lambda row: f"{row['source_dataset']}:{row['clip_id']}", axis=1
    )
print("Valid clips:", len(manifest), "Groups:", manifest.group_id.nunique())
display(manifest.canonical_class.value_counts().sort_index().rename("clips"))

V6C_TEST_PATH = Path("/content/v6c_test_window_predictions.csv")
download(
    f"{REPOSITORY_RAW}/v6c/model/test_window_predictions.csv",
    V6C_TEST_PATH,
)
V6C_TEST = pd.read_csv(V6C_TEST_PATH).set_index("window_key")
V6C_GATE_THRESHOLD = float(V6C_METRICS["gate_threshold"])
print("Frozen V6-C gate threshold:", V6C_GATE_THRESHOLD)


## Rebuild windows and load the frozen V6-A.1 split

The exported V6-A.1 `split_assignments.csv` is the authority. Every source
group receives exactly the same train, validation, or test assignment used by
V6-B and V6-C. The active taxonomy still maps archived plank to `unknown`.


In [ ]:
def resample_sequence(sequence, source_fps, target_fps=15):
    if not source_fps or abs(float(source_fps) - target_fps) < 0.1:
        return sequence
    duration = (len(sequence) - 1) / float(source_fps)
    target_length = max(2, int(round(duration * target_fps)) + 1)
    old_x = np.linspace(0, duration, len(sequence))
    new_x = np.linspace(0, duration, target_length)
    flat = sequence.reshape(len(sequence), -1)
    sampled = np.stack([
        np.interp(new_x, old_x, flat[:, index])
        for index in range(flat.shape[1])
    ], axis=1)
    return sampled.reshape(target_length, 17, 4).astype(np.float32)


windows, original_labels, groups = [], [], []
clip_ids, sources, starts = [], [], []
for row in manifest.itertuples(index=False):
    with np.load(row.npz_path, allow_pickle=False) as item:
        sequence = item["landmarks"].astype(np.float32)
    if sequence.ndim != 3 or sequence.shape[1:] != (17, 4):
        print("Skipping unexpected shape:", row.npz_path, sequence.shape)
        continue
    sequence = resample_sequence(sequence, getattr(row, "source_fps", 15) or 15)
    clip_starts = list(range(0, max(1, len(sequence) - WINDOW + 1), STRIDE))
    if len(sequence) > WINDOW and clip_starts[-1] != len(sequence) - WINDOW:
        clip_starts.append(len(sequence) - WINDOW)
    for start in clip_starts:
        window = sequence[start:start + WINDOW]
        if len(window) < WINDOW:
            window = np.concatenate([
                window, np.repeat(window[-1:], WINDOW - len(window), axis=0)
            ])
        windows.append(window)
        original_labels.append(row.canonical_class)
        groups.append(row.group_id)
        clip_ids.append(row.clip_id)
        sources.append(row.source_dataset)
        starts.append(start)

X = np.asarray(windows, dtype=np.float32)
original_labels = np.asarray(original_labels)
groups = np.asarray(groups)
clip_ids = np.asarray(clip_ids)
sources = np.asarray(sources)
starts = np.asarray(starts, dtype=np.int64)
window_keys = np.asarray([
    f"{source}|{group}|{clip}|{start}"
    for source, group, clip, start in zip(sources, groups, clip_ids, starts)
])

# Reuse the exact group assignment exported by V6-A.1.
frozen_split = pd.read_csv(V6A1_SPLIT_PATH)
assert frozen_split.groupby("group_id")["split"].nunique().max() == 1
group_to_split = (
    frozen_split[["group_id", "split"]]
    .drop_duplicates("group_id")
    .set_index("group_id")["split"]
    .to_dict()
)
assigned_split = np.asarray([group_to_split.get(group, "") for group in groups])
known_split_mask = assigned_split != ""
missing_groups = set(groups[~known_split_mask])
assert missing_groups <= UNREVIEWED_GROUPS, (
    "Unexpected groups absent from the frozen split: " + str(sorted(missing_groups))
)
train_idx = np.flatnonzero(assigned_split == "train")
val_idx = np.flatnonzero(assigned_split == "validation")
test_idx = np.flatnonzero(assigned_split == "test")

labels = np.where(np.isin(original_labels, list(RETIRED_CLASSES)), "unknown", original_labels)
ALL_CLASSES = sorted(np.unique(labels).tolist())
SUPPORTED_CLASSES = [name for name in ALL_CLASSES if name != "unknown"]
assert len(SUPPORTED_CLASSES) == 14, SUPPORTED_CLASSES
all_to_id = {name: index for index, name in enumerate(ALL_CLASSES)}
supported_to_id = {name: index for index, name in enumerate(SUPPORTED_CLASSES)}
y_all = np.asarray([all_to_id[name] for name in labels], dtype=np.int64)
y_gate = (labels != "unknown").astype(np.int64)
y_exercise = np.asarray([supported_to_id.get(name, -1) for name in labels], dtype=np.int64)

assert not (set(groups[train_idx]) & set(groups[val_idx]))
assert not (set(groups[train_idx]) & set(groups[test_idx]))
assert not (set(groups[val_idx]) & set(groups[test_idx]))
assert not (set(groups[train_idx]) & UNREVIEWED_GROUPS)
assert set(window_keys[known_split_mask]) == set(frozen_split.window_key), (
    "Reviewed window evidence changed since V6-A.1."
)
print("Excluded unresolved groups:", sorted(missing_groups))

print("Windows:", X.shape)
print("Frozen groups:", len(group_to_split))
print("Active supported classes:", SUPPORTED_CLASSES)
print("Retired plank windows mapped to unknown:", int((original_labels == "plank").sum()))
for name, indices in [("train", train_idx), ("validation", val_idx), ("test", test_idx)]:
    print(name, "windows=", len(indices), "groups=", len(np.unique(groups[indices])))
    display(pd.Series(labels[indices]).value_counts().sort_index().rename("windows"))


## Tempered hierarchical sampling and hard-positive weighting

Sampling first assigns equal probability to each class. Inside a class, each
source group receives probability proportional to `window_count ** 0.5`. This is
the midpoint between failed V6-A equal-group sampling (`** 0`) and raw window
sampling (`** 1`): long videos are restrained without making a one-window group
as influential as a large group. Notebook 13 hard-positive modifiers are
normalized inside the same class/group, so they change which windows are seen
without changing that group's total probability.

For the gate, “class” means supported or unknown. For the exercise model it
means one of the 14 active movements. `WeightedRandomSampler` then draws the
requested class -> group -> window distribution.


In [ ]:
hard_weight_by_key = dict(zip(
    hard_positive_frame.window_key,
    hard_positive_frame.recommended_sample_weight,
))
hard_positive_weights = np.asarray([
    float(hard_weight_by_key.get(key, 1.0)) for key in window_keys
], dtype=np.float64)
# Retired plank is now unknown and cannot receive a supported hard-positive boost.
hard_positive_weights[labels == "unknown"] = 1.0
matched_training_keys = (
    set(window_keys[train_idx][labels[train_idx] != "unknown"])
    & set(hard_weight_by_key)
)
assert not (set(window_keys[val_idx]) & set(hard_weight_by_key)), (
    "Notebook 13 weights unexpectedly intersect validation."
)
assert not (set(window_keys[test_idx]) & set(hard_weight_by_key)), (
    "Notebook 13 weights unexpectedly intersect test."
)
print("Notebook 13 keys:", len(hard_weight_by_key))
print("Matched active training keys:", len(matched_training_keys))
print("Upweighted active training windows:", int((hard_positive_weights[train_idx] > 1).sum()))


def hierarchical_window_weights(indices, hierarchy_labels, extra_weights=None):
    table = pd.DataFrame({
        "index": indices,
        "hierarchy_label": hierarchy_labels[indices],
        "group": groups[indices],
    })
    class_count = table["hierarchy_label"].nunique()
    group_sizes = table.groupby(
        ["hierarchy_label", "group"]
    )["index"].transform("count").astype(float)
    table["group_mass"] = group_sizes.pow(GROUP_TEMPERATURE)
    group_table = table[["hierarchy_label", "group"]].copy()
    group_table["group_mass"] = table["group_mass"]
    class_group_mass = (
        group_table.drop_duplicates(["hierarchy_label", "group"])
        .groupby("hierarchy_label")["group_mass"].sum()
    )
    table["extra_weight"] = (
        1.0 if extra_weights is None else extra_weights[table["index"].to_numpy()]
    )
    table["pair_extra_total"] = table.groupby(
        ["hierarchy_label", "group"]
    )["extra_weight"].transform("sum")
    weights = []
    for row in table.itertuples(index=False):
        weight = (
            1.0 / class_count
            * row.group_mass / class_group_mass.loc[row.hierarchy_label]
            * row.extra_weight / row.pair_extra_total
        )
        weights.append(weight)
    weights = np.asarray(weights, dtype=np.float64)
    return weights / weights.mean()


# Use strings here so the audit table remains human-readable.
gate_hierarchy = np.where(y_gate == 1, "supported", "unknown")
gate_train_weights = hierarchical_window_weights(
    train_idx, gate_hierarchy, hard_positive_weights
)
exercise_train_idx = train_idx[y_exercise[train_idx] >= 0]
exercise_val_idx = val_idx[y_exercise[val_idx] >= 0]
exercise_test_idx = test_idx[y_exercise[test_idx] >= 0]
exercise_train_weights = hierarchical_window_weights(
    exercise_train_idx, labels
)

sampler_audit = pd.DataFrame({
    "window_key": window_keys[train_idx],
    "canonical_class": labels[train_idx],
    "original_class": original_labels[train_idx],
    "group_id": groups[train_idx],
    "gate_target": gate_hierarchy[train_idx],
    "hard_positive_weight": hard_positive_weights[train_idx],
    "gate_sampler_weight": gate_train_weights,
})
exercise_weight_map = dict(zip(
    window_keys[exercise_train_idx], exercise_train_weights
))
sampler_audit["exercise_sampler_weight"] = sampler_audit.window_key.map(
    exercise_weight_map
).fillna(0.0)

display(sampler_audit.groupby("gate_target").agg(
    windows=("window_key", "count"),
    groups=("group_id", "nunique"),
    total_sampler_weight=("gate_sampler_weight", "sum"),
).round(3))
display(sampler_audit[sampler_audit.canonical_class != "unknown"].groupby(
    "canonical_class"
).agg(
    windows=("window_key", "count"),
    groups=("group_id", "nunique"),
    total_exercise_weight=("exercise_sampler_weight", "sum"),
).round(3))


In [ ]:
LEFT_RIGHT_PAIRS = [
    (1, 2), (3, 4), (5, 6), (7, 8),
    (9, 10), (11, 12), (13, 14), (15, 16),
]


def augment_pose_window(sample, force_all=False):
    sample = sample.copy()
    config = AUGMENTATION_CONFIG

    if force_all or np.random.random() < config["horizontal_flip_probability"]:
        sample[:, :, 0] *= -1.0
        for left, right in LEFT_RIGHT_PAIRS:
            sample[:, [left, right], :] = sample[:, [right, left], :]

    if force_all or np.random.random() < config["rotation_probability"]:
        degrees = np.random.uniform(
            -config["max_rotation_degrees"], config["max_rotation_degrees"]
        )
        radians = np.deg2rad(degrees)
        cosine, sine = np.cos(radians), np.sin(radians)
        x = sample[:, :, 0].copy()
        y = sample[:, :, 1].copy()
        sample[:, :, 0] = cosine * x - sine * y
        sample[:, :, 1] = sine * x + cosine * y

    if force_all or np.random.random() < config["speed_probability"]:
        speed = np.random.uniform(*config["speed_range"])
        center = (WINDOW - 1) / 2.0
        positions = (np.arange(WINDOW) - center) * speed + center
        flat = sample.reshape(WINDOW, -1)
        flat = np.stack([
            np.interp(positions, np.arange(WINDOW), flat[:, column])
            for column in range(flat.shape[1])
        ], axis=1)
        sample = flat.reshape(WINDOW, 17, 4).astype(np.float32)

    if force_all or np.random.random() < config["frame_drop_probability"]:
        fraction = np.random.uniform(*config["frame_drop_fraction_range"])
        count = max(1, int(round(WINDOW * fraction)))
        dropped = np.random.choice(np.arange(1, WINDOW), size=count, replace=False)
        sample[dropped] = sample[dropped - 1]

    if force_all or np.random.random() < config["coordinate_noise_probability"]:
        sample[:, :, :2] += np.random.normal(
            0, config["coordinate_noise_std"], sample[:, :, :2].shape
        ).astype(np.float32)
    return sample.astype(np.float32)



NODE_FEATURE_CONFIG = {
    "coordinates_xy_per_joint": 2,
    "velocity_xy_per_joint": 2,
}
NODE_FEATURES = sum(NODE_FEATURE_CONFIG.values())


def pose_to_graph_features(sample):
    """Convert [time, 17, 4] pose to [channels, time, joints]."""
    xy = sample[:, :, :2].astype(np.float32)
    velocity = np.zeros_like(xy)
    velocity[1:] = xy[1:] - xy[:-1]
    features = np.concatenate([xy, velocity], axis=2).transpose(2, 0, 1)
    assert features.shape == (NODE_FEATURES, WINDOW, 17), features.shape
    assert np.isfinite(features).all()
    return features.astype(np.float32)


class PoseWindowDataset(Dataset):
    def __init__(self, indices, targets, training=False):
        self.indices = np.asarray(indices)
        self.targets = targets
        self.training = training

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        actual = self.indices[item]
        sample = X[actual].copy()
        if self.training:
            sample = augment_pose_window(sample)
        features = pose_to_graph_features(sample)
        return torch.from_numpy(features), torch.tensor(
            self.targets[actual], dtype=torch.long
        )


def make_loaders(targets, train_indices, val_indices, test_indices, train_weights, seed):
    generator = torch.Generator().manual_seed(seed)
    sampler = WeightedRandomSampler(
        torch.as_tensor(train_weights, dtype=torch.double),
        len(train_weights), replacement=True, generator=generator,
    )
    return (
        DataLoader(
            PoseWindowDataset(train_indices, targets, True),
            batch_size=BATCH_SIZE, sampler=sampler,
        ),
        DataLoader(
            PoseWindowDataset(val_indices, targets),
            batch_size=BATCH_SIZE, shuffle=False,
        ),
        DataLoader(
            PoseWindowDataset(test_indices, targets),
            batch_size=BATCH_SIZE, shuffle=False,
        ),
    )


gate_loaders = make_loaders(
    y_gate, train_idx, val_idx, test_idx, gate_train_weights, SEED
)
exercise_loaders = make_loaders(
    y_exercise, exercise_train_idx, exercise_val_idx, exercise_test_idx,
    exercise_train_weights, SEED + 1,
)
print("Node feature blocks:", NODE_FEATURE_CONFIG)
print("Input channels per joint:", NODE_FEATURES)
print("Gate split sizes:", [len(loader.dataset) for loader in gate_loaders])
print("Exercise split sizes:", [len(loader.dataset) for loader in exercise_loaders])


## Audit the augmentation before training

The preview applies every transformation once to a real training window. Check
that limbs remain connected, left/right swapping is anatomical, and motion is
perturbed without becoming a different exercise. The image is included in the
report for reproducibility.


In [ ]:
SKELETON_EDGES = [
    (5, 6), (5, 7), (7, 9), (6, 8), (8, 10),
    (5, 11), (6, 12), (11, 12),
    (11, 13), (13, 15), (12, 14), (14, 16),
]
preview_candidates = train_idx[labels[train_idx] == "squat"]
preview_index = int(preview_candidates[0] if len(preview_candidates) else train_idx[0])
original_preview = X[preview_index].copy()
random_state = np.random.get_state()
np.random.seed(SEED + 1000)
augmented_preview = augment_pose_window(original_preview, force_all=True)
np.random.set_state(random_state)

fig, axes = plt.subplots(2, 6, figsize=(15, 6))
frame_indices = np.linspace(0, WINDOW - 1, 6, dtype=int)
for row, (title, sequence) in enumerate([
    ("original", original_preview), ("augmented", augmented_preview)
]):
    for axis, frame_index in zip(axes[row], frame_indices):
        xy = sequence[frame_index, :, :2]
        for left, right in SKELETON_EDGES:
            axis.plot(
                [xy[left, 0], xy[right, 0]],
                [-xy[left, 1], -xy[right, 1]],
                color="tab:blue", linewidth=1.4,
            )
        axis.scatter(xy[:, 0], -xy[:, 1], s=12, color="tab:red")
        axis.set_aspect("equal")
        axis.axis("off")
        axis.set_title(f"{title} t={frame_index}")
fig.suptitle(f"V6-B augmentation audit | {labels[preview_index]} | {window_keys[preview_index]}")
fig.tight_layout()
AUGMENTATION_PREVIEW_PATH = Path("/content/v6b_augmentation_preview.png")
fig.savefig(AUGMENTATION_PREVIEW_PATH, dpi=170, bbox_inches="tight")
plt.show()
print("Preview:", AUGMENTATION_PREVIEW_PATH)


## Three controlled classifier families


In [ ]:
SKELETON_EDGES = [
    (5, 6), (5, 7), (7, 9), (6, 8), (8, 10),
    (5, 11), (6, 12), (11, 12),
    (11, 13), (13, 15), (12, 14), (14, 16),
]


def normalized_adjacency(num_nodes=17):
    adjacency = np.eye(num_nodes, dtype=np.float32)
    for first, second in SKELETON_EDGES:
        adjacency[first, second] = adjacency[second, first] = 1.0
    degree = adjacency.sum(axis=1)
    inverse_sqrt = np.diag(1.0 / np.sqrt(np.maximum(degree, 1e-6)))
    return (inverse_sqrt @ adjacency @ inverse_sqrt).astype(np.float32)


class BiGRUClassifier(nn.Module):
    def __init__(self, num_classes, hidden=96):
        super().__init__()
        self.input_projection = nn.Sequential(nn.Linear(17 * NODE_FEATURES, 128), nn.LayerNorm(128), nn.ReLU())
        self.gru = nn.GRU(128, hidden, num_layers=2, batch_first=True, bidirectional=True, dropout=0.2)
        self.classifier = nn.Sequential(nn.LayerNorm(hidden * 4), nn.Dropout(0.25), nn.Linear(hidden * 4, num_classes))
    def forward(self, x):
        x = x.permute(0, 2, 3, 1).reshape(x.size(0), x.size(2), -1)
        x, _ = self.gru(self.input_projection(x))
        pooled = torch.cat([x.mean(1), x.amax(1)], dim=1)
        return self.classifier(pooled)


class PoseTransformerClassifier(nn.Module):
    def __init__(self, num_classes, width=128, heads=4, layers=3):
        super().__init__()
        self.input_projection = nn.Linear(17 * NODE_FEATURES, width)
        self.position = nn.Parameter(torch.zeros(1, WINDOW, width))
        layer = nn.TransformerEncoderLayer(width, heads, width * 3, dropout=0.2, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, layers)
        self.classifier = nn.Sequential(nn.LayerNorm(width * 2), nn.Dropout(0.25), nn.Linear(width * 2, num_classes))
    def forward(self, x):
        x = x.permute(0, 2, 3, 1).reshape(x.size(0), x.size(2), -1)
        x = self.encoder(self.input_projection(x) + self.position[:, :x.size(1)])
        return self.classifier(torch.cat([x.mean(1), x.amax(1)], dim=1))


class AdaptiveGraphBlock(nn.Module):
    def __init__(self, in_channels, out_channels, temporal_kernel=9, dropout=0.2):
        super().__init__()
        self.register_buffer("base_adjacency", torch.from_numpy(normalized_adjacency()))
        self.learned_adjacency = nn.Parameter(torch.zeros(17, 17))
        self.q = nn.Conv2d(in_channels, max(8, out_channels // 4), 1)
        self.k = nn.Conv2d(in_channels, max(8, out_channels // 4), 1)
        self.spatial = nn.Conv2d(in_channels, out_channels, 1)
        self.temporal = nn.Sequential(
            nn.BatchNorm2d(out_channels), nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, (temporal_kernel, 1), padding=((temporal_kernel - 1)//2, 0)),
            nn.BatchNorm2d(out_channels), nn.Dropout(dropout),
        )
        self.residual = nn.Conv2d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()
    def forward(self, x):
        q = self.q(x).mean(2).transpose(1, 2)
        k = self.k(x).mean(2)
        dynamic = torch.softmax(torch.matmul(q, k) / np.sqrt(k.size(1)), dim=-1)
        adjacency = self.base_adjacency + torch.tanh(self.learned_adjacency)
        mixed = torch.einsum("bctv,vw->bctw", x, adjacency) + torch.einsum("bctv,bvw->bctw", x, dynamic)
        return torch.relu(self.temporal(self.spatial(mixed)) + self.residual(x))


class AdaptiveGCNClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        channels = [32, 64, 96, 128]
        blocks=[]; current=NODE_FEATURES
        for out in channels:
            blocks.append(AdaptiveGraphBlock(current, out)); current=out
        self.features=nn.Sequential(*blocks)
        self.classifier=nn.Linear(current, num_classes)
    def forward(self, x):
        return self.classifier(self.features(x).mean((2,3)))


MODEL_FACTORIES = {
    "bigru": lambda n: BiGRUClassifier(n),
    "pose_transformer": lambda n: PoseTransformerClassifier(n),
    "adaptive_gcn": lambda n: AdaptiveGCNClassifier(n),
}
for name, factory in MODEL_FACTORIES.items():
    model=factory(len(SUPPORTED_CLASSES))
    print(name, f"{sum(p.numel() for p in model.parameters()):,}", "parameters")


## Train with the same sampler, augmentation, optimizer, and early stopping


In [ ]:
def predict_model(model, loader):
    model.eval(); truths=[]; probabilities=[]
    with torch.no_grad():
        for features, targets in loader:
            logits=model(features.to(DEVICE))
            truths.extend(targets.numpy()); probabilities.extend(logits.softmax(1).cpu().numpy())
    probabilities=np.asarray(probabilities)
    return np.asarray(truths), probabilities.argmax(1), probabilities


def train_classifier(name, factory, seed):
    seed_everything(seed)
    model=factory(len(SUPPORTED_CLASSES)).to(DEVICE)
    criterion=nn.CrossEntropyLoss()
    optimizer=torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", patience=3, factor=0.5)
    best_path=Path(f"/content/best_g2_{name}.pt")
    best_f1=-1.0; stale=0; history=[]
    def epoch(loader, training):
        model.train(training); losses=[]; truths=[]; predictions=[]
        context=torch.enable_grad() if training else torch.no_grad()
        with context:
            for features, targets in loader:
                features=features.to(DEVICE); targets=targets.to(DEVICE)
                if training: optimizer.zero_grad(set_to_none=True)
                logits=model(features); loss=criterion(logits, targets)
                if training: loss.backward(); optimizer.step()
                losses.append(loss.item()*len(targets)); truths.extend(targets.cpu().numpy()); predictions.extend(logits.argmax(1).cpu().numpy())
        return sum(losses)/len(truths), f1_score(truths,predictions,average="macro",zero_division=0)
    for number in range(1,MAX_EPOCHS+1):
        train_loss,train_f1=epoch(exercise_loaders[0],True)
        val_loss,val_f1=epoch(exercise_loaders[1],False); scheduler.step(val_f1)
        history.append({"model":name,"epoch":number,"train_loss":train_loss,"train_macro_f1":train_f1,"val_loss":val_loss,"val_macro_f1":val_f1})
        print(f"{name} {number:02d} train={train_f1:.3f} val={val_f1:.3f}")
        if val_f1>best_f1+1e-4:
            best_f1=val_f1; stale=0; torch.save(model.state_dict(),best_path)
        else:
            stale+=1
            if stale>=PATIENCE: break
    model.load_state_dict(torch.load(best_path,map_location=DEVICE))
    return model,history,best_f1

models={}; histories=[]; validation_scores={}
for offset,(name,factory) in enumerate(MODEL_FACTORIES.items()):
    model,history,best=train_classifier(name,factory,SEED+20+offset)
    models[name]=model; histories.extend(history); validation_scores[name]=best
print("Best validation exercise Macro F1:",validation_scores)


## Compare with the frozen V6-C gate and export


In [ ]:
# Build deterministic all-test features once for end-to-end hybrids.
all_test_loader = DataLoader(PoseWindowDataset(test_idx, y_all), batch_size=128, shuffle=False)
true_all = labels[test_idx]
all_test_keys = window_keys[test_idx]
missing = [key for key in all_test_keys if key not in V6C_TEST.index]
assert not missing, f"V6-C gate predictions missing {len(missing)} test windows"
v6c_gate = np.asarray([V6C_TEST.loc[key, "gate_supported_probability"] for key in all_test_keys], dtype=float)

rows=[]; model_probabilities={}; end_predictions={}
for name,model in models.items():
    _,_,oracle_probability=predict_model(model,exercise_loaders[2])
    oracle_truth=labels[exercise_test_idx]
    oracle_prediction=np.asarray([SUPPORTED_CLASSES[i] for i in oracle_probability.argmax(1)])
    _,_,all_probability=predict_model(model,all_test_loader)
    model_probabilities[name]=all_probability
    supported_prediction=np.asarray([SUPPORTED_CLASSES[i] for i in all_probability.argmax(1)])
    final=np.where(v6c_gate>=V6C_GATE_THRESHOLD,supported_prediction,"unknown")
    end_predictions[name]=final
    rows.append({
        "model":name,"validation_exercise_macro_f1":validation_scores[name],
        "oracle_exercise_macro_f1":f1_score(oracle_truth,oracle_prediction,labels=SUPPORTED_CLASSES,average="macro",zero_division=0),
        "end_to_end_macro_f1":f1_score(true_all,final,labels=ALL_CLASSES,average="macro",zero_division=0),
        "end_to_end_balanced_accuracy":balanced_accuracy_score(true_all,final),
        "end_to_end_accuracy":accuracy_score(true_all,final),
    })

# Equal averaging is fixed before viewing test metrics; no test-derived weights.
ensemble_probability=np.mean(np.stack(list(model_probabilities.values())),axis=0)
ensemble_supported=np.asarray([SUPPORTED_CLASSES[i] for i in ensemble_probability.argmax(1)])
ensemble_final=np.where(v6c_gate>=V6C_GATE_THRESHOLD,ensemble_supported,"unknown")
end_predictions["equal_ensemble"]=ensemble_final
rows.append({
    "model":"equal_ensemble","validation_exercise_macro_f1":float(np.mean(list(validation_scores.values()))),
    "oracle_exercise_macro_f1":np.nan,
    "end_to_end_macro_f1":f1_score(true_all,ensemble_final,labels=ALL_CLASSES,average="macro",zero_division=0),
    "end_to_end_balanced_accuracy":balanced_accuracy_score(true_all,ensemble_final),
    "end_to_end_accuracy":accuracy_score(true_all,ensemble_final),
})
comparison=pd.DataFrame(rows).sort_values("end_to_end_macro_f1",ascending=False)
display(comparison)

REPORT_ROOT=Path("/content/g2_pose_model_suite_report")
if REPORT_ROOT.exists(): shutil.rmtree(REPORT_ROOT)
REPORT_ROOT.mkdir()
comparison.to_csv(REPORT_ROOT/"model_comparison.csv",index=False)
pd.DataFrame(histories).to_csv(REPORT_ROOT/"training_history.csv",index=False)
sampler_audit.to_csv(REPORT_ROOT/"sampler_audit.csv",index=False)
for name,model in models.items():
    torch.save({"state_dict":model.state_dict(),"model":name,"supported_classes":SUPPORTED_CLASSES,"window_frames":WINDOW,"node_features":NODE_FEATURE_CONFIG},REPORT_ROOT/f"{name}.pt")
for name,prediction in end_predictions.items():
    report=classification_report(true_all,prediction,labels=ALL_CLASSES,output_dict=True,zero_division=0)
    (REPORT_ROOT/f"{name}_classification_report.json").write_text(json.dumps(report,indent=2))
    pd.DataFrame({"window_key":all_test_keys,"true_class":true_all,"v6c_gate_probability":v6c_gate,"predicted_class":prediction}).to_csv(REPORT_ROOT/f"{name}_test_predictions.csv",index=False)
    matrix=confusion_matrix(true_all,prediction,labels=ALL_CLASSES)
    plt.figure(figsize=(12,10)); sns.heatmap(matrix,annot=True,fmt="d",cmap="Blues",xticklabels=ALL_CLASSES,yticklabels=ALL_CLASSES)
    plt.title(name+" with frozen V6-C gate"); plt.tight_layout(); plt.savefig(REPORT_ROOT/f"{name}_confusion_matrix.png",dpi=160); plt.close()
metrics={"experiment":"g2_controlled_pose_model_suite","frozen_gate":"v6c","gate_threshold":V6C_GATE_THRESHOLD,"models":rows,"selection_rule":"validation Macro F1; test metrics reporting only"}
(REPORT_ROOT/"metrics.json").write_text(json.dumps(metrics,indent=2))
zip_path=shutil.make_archive("/content/calorie_g2_pose_model_suite_report","zip",root_dir=REPORT_ROOT)
print("Report:",zip_path,"MB",round(Path(zip_path).stat().st_size/1024**2,2))
files.download(zip_path)


## Interpretation rules

- Do not pick a model because of test accuracy alone.
- First inspect validation Macro F1 and per-class recall.
- The V6-C gate is frozen, so differences come from the exercise classifier.
- The equal ensemble uses no test-derived weights.
- A candidate must later survive creator-separated validation and long-video evaluation.
